 ### Batch Purge CSV Rows by Date

 Scans `C:\Users\ping\Desktop\yloader` (non-recursive) and deletes any row matching the target date.

In [ ]:
import csv
import os
from pathlib import Path
import tempfile
from datetime import datetime


def parse_cell_date(val: str):
    """Parse dates formatted as YYYY-MM-DD, M/D/YYYY, MM/DD/YYYY, etc."""
    val = val.strip()
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%Y/%m/%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(val, fmt).date()
        except ValueError:
            continue

    # Fallback for variable non-padded formats like 9/25/2026
    parts = val.replace("-", "/").split("/")
    if len(parts) == 3:
        try:
            if len(parts[0]) == 4:  # YYYY/M/D
                return datetime(int(parts[0]), int(parts[1]), int(parts[2])).date()
            else:  # M/D/YYYY
                return datetime(int(parts[2]), int(parts[0]), int(parts[1])).date()
        except (ValueError, TypeError):
            pass
    return None


def purge_date_from_csvs(
    target_dir: str | Path,
    target_date_str: str = "2026-09-25",
) -> None:
    # Strip hidden Windows unicode markers (\u202a-\u202e) from copy-pasting paths
    clean_path = str(target_dir).strip("\u202a\u202b\u202c\u202d\u202e ")
    folder = Path(clean_path)

    assert folder.exists() and folder.is_dir(), f"Directory not found: {folder}"

    target_date = datetime.strptime(target_date_str.strip(), "%Y-%m-%d").date()
    print(f"Target Purge Date : {target_date}")
    print(f"Target Directory  : {folder}\n")

    csv_files = [
        f for f in folder.iterdir() if f.is_file() and f.suffix.lower() == ".csv"
    ]
    print(f"Discovered {len(csv_files)} CSV files. Processing...")

    total_deleted = 0
    modified_files = 0

    for file_path in csv_files:
        deleted_in_file = 0
        rows_to_keep = []

        with open(file_path, mode="r", newline="", encoding="utf-8-sig") as infile:
            reader = csv.reader(infile)
            for row in reader:
                if not row:
                    continue
                first_col = row[0].strip()
                cell_date = parse_cell_date(first_col)

                if cell_date == target_date:
                    deleted_in_file += 1
                else:
                    rows_to_keep.append(row)

        if deleted_in_file > 0:
            # Atomic rewrite via tempfile on same disk avoids corrupting data if interrupted
            with tempfile.NamedTemporaryFile(
                mode="w",
                newline="",
                encoding="utf-8",
                dir=folder,
                delete=False,
            ) as tmp_file:
                writer = csv.writer(tmp_file)
                writer.writerows(rows_to_keep)
                temp_name = tmp_file.name

            os.replace(temp_name, file_path)
            total_deleted += deleted_in_file
            modified_files += 1
            print(f"  [MODIFIED] {file_path.name}: removed {deleted_in_file} row(s)")

    print(f"\nCompleted. Purged {total_deleted} rows across {modified_files} files.")

In [ ]:
# Execution parameters
TARGET_DIRECTORY = r"C:\Users\ping\Desktop\yloader"
INPUT_DATE = "2026-09-25"

purge_date_from_csvs(TARGET_DIRECTORY, INPUT_DATE)

Target Purge Date : 2026-09-25
Target Directory  : C:\Users\ping\Desktop\yloader

Discovered 1725 CSV files. Processing...
  [MODIFIED] A.csv: removed 1 row(s)
  [MODIFIED] AA.csv: removed 1 row(s)
  [MODIFIED] AAL.csv: removed 1 row(s)
  [MODIFIED] AAON.csv: removed 1 row(s)
  [MODIFIED] AAP.csv: removed 1 row(s)
  [MODIFIED] AAPL.csv: removed 1 row(s)
  [MODIFIED] ABBV.csv: removed 1 row(s)
  [MODIFIED] ABEV.csv: removed 1 row(s)
  [MODIFIED] ABNB.csv: removed 1 row(s)
  [MODIFIED] ABT.csv: removed 1 row(s)
  [MODIFIED] ABVX.csv: removed 1 row(s)
  [MODIFIED] ACGL.csv: removed 1 row(s)
  [MODIFIED] ACI.csv: removed 1 row(s)
  [MODIFIED] ACM.csv: removed 1 row(s)
  [MODIFIED] ACN.csv: removed 1 row(s)
  [MODIFIED] ACT.csv: removed 1 row(s)
  [MODIFIED] ACWI.csv: removed 1 row(s)
  [MODIFIED] ACWX.csv: removed 1 row(s)
  [MODIFIED] ADBE.csv: removed 1 row(s)
  [MODIFIED] ADC.csv: removed 1 row(s)
  [MODIFIED] ADI.csv: removed 1 row(s)
  [MODIFIED] ADM.csv: removed 1 row(s)
  [MODIFIED]